In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import pandas as pd
from src.data.ingestion.statcast_client import project_root
from src.utils.leakage import (
    banned_for_pitch_outcome, safe_features, describe_exclusions, check_features
)

raw = project_root() / "data" / "raw"
df = pd.concat([pd.read_parquet(p) for p in sorted(raw.glob("statcast_*.parquet"))],
               ignore_index=True)

banned = banned_for_pitch_outcome()
print(f"total columns: {len(df.columns)}")
print(f"banned:        {len(set(df.columns) & banned)}")
print(f"available:     {len(df.columns) - len(set(df.columns) & banned)}")

total columns: 119
banned:        41
available:     78


In [2]:
describe_exclusions(df, banned)

,column,reason
0,break_angle_deprecated,empty in all inspected seasons
1,break_length_deprecated,empty in all inspected seasons
2,spin_rate_deprecated,empty in all inspected seasons
3,tfs_deprecated,empty in all inspected seasons
4,tfs_zulu_deprecated,empty in all inspected seasons
5,umpire,empty in all inspected seasons
6,at_bat_number,"identifier or bookkeeping, not a feature"
7,game_date,"identifier or bookkeeping, not a feature"
8,game_pk,"identifier or bookkeeping, not a feature"
9,game_type,"identifier or bookkeeping, not a feature"


In [3]:
X = safe_features(df, banned, context="whiff model")
print(X.shape)

# 일부러 누수를 넣어보기
try:
    check_features(list(X.columns) + ["launch_speed"], banned, context="whiff model")
except Exception as e:
    print("CAUGHT:", e)

(10654, 78)
CAUGHT: whiff model: 1 banned column(s) in the feature matrix: ['launch_speed']. These are not knowable at prediction time.
